In [1]:
import pickle 
dataset = "hopper-medium-expert-v2"
env_name = dataset.split("-")[0][0].upper() + dataset.split("-")[0][1:] + "-v4"
print(env_name)

pkl_path = f"../../../../master_thesis/reproducing_decision_transformer/gymnasium/data/{dataset}.pkl"
with open(pkl_path, "rb") as f:
    episodes = pickle.load(f)

print(type(episodes))
print(len(episodes))
print(episodes[0].keys())
print(episodes[0]["observations"].shape)


Hopper-v4
<class 'list'>
3213
dict_keys(['observations', 'next_observations', 'actions', 'rewards', 'terminals'])
(470, 11)


In [2]:
import numpy as np
import pickle
from d3rlpy.dataset.components import Episode  # Ensure this imports the default Episode class

def load_and_convert_episodes(pkl_path):
    # Load your episodes from the pickle file
    with open(pkl_path, "rb") as f:
        raw_episodes = pickle.load(f)
    
    converted_episodes = []
    for ep in raw_episodes:
        # Assume each ep is a dict with keys as shown in your printout.
        observations = ep["observations"]           # shape (470, 11)
        actions = ep["actions"]
        rewards = ep["rewards"]
        terminals = ep["terminals"]
        if isinstance(terminals, (list, np.ndarray)):
            terminated = bool(terminals[-1])
        else:
            terminated = bool(terminals)
        # Optionally, next_observations is available as well.
        next_observations = ep.get("next_observations", None)
        
        # Create an Episode object.
        # The Episode class in d3rlpy typically accepts these arrays directly.
        episode = Episode(
            observations=observations,
            actions=actions,
            rewards=rewards,
            terminated=terminated,
        )
        converted_episodes.append(episode)
    
    return converted_episodes

# Example usage:
episodes = load_and_convert_episodes(pkl_path)
print(f"Converted {len(episodes)} episodes.")

# Now, to create a ReplayBuffer with these episodes:
from d3rlpy.dataset import ReplayBuffer, FIFOBuffer

buffer_impl = FIFOBuffer(limit=1000000)
replay_buffer = ReplayBuffer(buffer=buffer_impl, episodes=episodes)

/gpfs/data/fs72297/jklotz/.conda/envs/d3rlpy_dev_requirements_py310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Converted 3213 episodes.
2025-06-21 12:55.58 [info     ] Signatures have been automatically determined. action_signature=Signature(dtype=[dtype('float32')], shape=[(3,)]) observation_signature=Signature(dtype=[dtype('float32')], shape=[(11,)]) reward_signature=Signature(dtype=[dtype('float32')], shape=[()])
2025-06-21 12:55.58 [info     ] Action-space has been automatically determined. action_space=<ActionSpace.CONTINUOUS: 1>
2025-06-21 12:55.58 [info     ] Action size has been automatically determined. action_size=3


In [3]:
import numpy as np
import pickle
from d3rlpy.dataset.components import Episode

def convert_raw_episode(raw_ep):
    # Convert to NumPy arrays if they aren't already.
    observations = np.array(raw_ep["observations"])
    actions = np.array(raw_ep["actions"])
    rewards = np.array(raw_ep["rewards"])
    terminals = raw_ep["terminals"]

    # Make sure rewards is 2D: shape (T, 1) rather than (T,)
    if rewards.ndim == 1:
        rewards = rewards.reshape(-1, 1)
    
    # For the termination flag, take the last element of terminals.
    if isinstance(terminals, (list, np.ndarray)):
        terminated = bool(terminals[-1])
    else:
        terminated = bool(terminals)
    
    return Episode(
        observations=observations,
        actions=actions,
        rewards=rewards,
        terminated=terminated
    )

def load_and_convert_episodes(pkl_path):
    with open(pkl_path, "rb") as f:
        raw_episodes = pickle.load(f)
    return [convert_raw_episode(ep) for ep in raw_episodes]

# Example usage:
episodes = load_and_convert_episodes(pkl_path=pkl_path)
print(f"Loaded {len(episodes)} episodes.")

from d3rlpy.dataset import ReplayBuffer, FIFOBuffer

buffer_impl = FIFOBuffer(limit=1000000)
replay_buffer = ReplayBuffer(buffer=buffer_impl, episodes=episodes)

Loaded 3213 episodes.
2025-06-21 12:56.01 [info     ] Signatures have been automatically determined. action_signature=Signature(dtype=[dtype('float32')], shape=[(3,)]) observation_signature=Signature(dtype=[dtype('float32')], shape=[(11,)]) reward_signature=Signature(dtype=[dtype('float32')], shape=[(1,)])
2025-06-21 12:56.01 [info     ] Action-space has been automatically determined. action_space=<ActionSpace.CONTINUOUS: 1>
2025-06-21 12:56.01 [info     ] Action size has been automatically determined. action_size=3


In [1]:
import numpy as np
import pickle
from d3rlpy.dataset.components import Episode

dataset = "hopper-medium-expert-v2"
if "halfcheetah" in dataset:
    env_name = "HalfCheetah-v4"
elif "hopper" in dataset:
    env_name = "Hopper-v4"
elif "walker" in dataset:
    env_name = "Walker2d-v4"

#env_name = dataset.split("-")[0][0].upper() + dataset.split("-")[0][1:] + "-v5"
print(env_name)

pkl_path = f"../../../../master_thesis/reproducing_decision_transformer/gymnasium/data/{dataset}.pkl"

def convert_raw_episode(raw_ep):
    # Convert raw observations to a NumPy array and then to a list of individual observations.
    observations = np.array(raw_ep["observations"])

    # Ensure actions and rewards are NumPy arrays.
    actions = np.array(raw_ep["actions"])
    rewards = np.array(raw_ep["rewards"])
    # For rewards, ensure they have an extra dimension (T, 1)
    if rewards.ndim == 1:
        rewards = rewards.reshape(-1, 1)
    
    # Use the last element of "terminals" as the terminated flag.
    terminals = raw_ep["terminals"]
    if isinstance(terminals, (list, np.ndarray)):
        terminated = bool(terminals[-1])
    else:
        terminated = bool(terminals)
    
    return Episode(
        observations=observations,
        actions=actions,
        rewards=rewards,
        terminated=terminated
    )

def load_and_convert_episodes(pkl_path):
    with open(pkl_path, "rb") as f:
        raw_episodes = pickle.load(f)
    return [convert_raw_episode(ep) for ep in raw_episodes]

# Example usage:
episodes = load_and_convert_episodes(pkl_path=pkl_path)
print(f"Loaded {len(episodes)} episodes.")

from d3rlpy.dataset import ReplayBuffer, FIFOBuffer

buffer_impl = FIFOBuffer(limit=10000000)
replay_buffer = ReplayBuffer(buffer=buffer_impl, episodes=episodes)


/gpfs/data/fs72297/jklotz/.conda/envs/d3rlpy_dev_requirements_py310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Hopper-v4
Loaded 3213 episodes.
2025-06-21 13:05.47 [info     ] Signatures have been automatically determined. action_signature=Signature(dtype=[dtype('float32')], shape=[(3,)]) observation_signature=Signature(dtype=[dtype('float32')], shape=[(11,)]) reward_signature=Signature(dtype=[dtype('float32')], shape=[(1,)])
2025-06-21 13:05.47 [info     ] Action-space has been automatically determined. action_space=<ActionSpace.CONTINUOUS: 1>
2025-06-21 13:05.47 [info     ] Action size has been automatically determined. action_size=3


In [2]:
import gymnasium as gym
import d3rlpy
import argparse
# parser.add_argument("--dataset", type=str, default="hopper-medium-v0")
# parser.add_argument("--seed", type=int, default=1)
# parser.add_argument("--gpu", type=int)
# parser.add_argument("--compile", action="store_true")
# args = parser.parse_args()

args = argparse.Namespace()
args.dataset = dataset
args.seed = 1
args.gpu = 1
args.compile = False

import gym
env = gym.make(env_name)

#dataset, env = d3rlpy.datasets.get_dataset(args.dataset)

# fix seed
d3rlpy.seed(args.seed)
d3rlpy.envs.seed_env(env, args.seed)

if "halfcheetah" in args.dataset:
    target_return = 6000
elif "hopper" in args.dataset:
    target_return = 3600
elif "walker" in args.dataset:
    target_return = 5000
else:
    raise ValueError("unsupported dataset")

In [3]:
print(env.observation_space)

Box(-inf, inf, (11,), float64)


In [ ]:
dt = d3rlpy.algos.DecisionTransformerConfig(
    batch_size=64,
    learning_rate=1e-4,
    optim_factory=d3rlpy.optimizers.AdamWFactory(
        weight_decay=1e-4,
        clip_grad_norm=0.25,
        lr_scheduler_factory=d3rlpy.optimizers.WarmupSchedulerFactory(
            warmup_steps=10000#10000
        ),
    ),
    encoder_factory=d3rlpy.models.VectorEncoderFactory(
        [128],
        exclude_last_activation=True,
    ),
    observation_scaler=d3rlpy.preprocessing.StandardObservationScaler(),
    reward_scaler=d3rlpy.preprocessing.MultiplyRewardScaler(0.001),
    position_encoding_type=d3rlpy.PositionEncodingType.SIMPLE,
    context_size=20,
    num_heads=1,
    num_layers=3,
    max_timestep=1000,
    compile_graph=args.compile,
).create(device="cuda")

dt.fit(
    replay_buffer,
    n_steps=100000,#100000,
    n_steps_per_epoch=1000,#1000,
    save_interval=10,
    eval_env=env,
    eval_target_return=target_return,
    experiment_name=f"DT_{args.dataset}_{args.seed}",
    n_trials=50,
    eval_gaps=5,
)

2025-06-21 13:05.51 [info     ] dataset info                   dataset_info=DatasetInfo(observation_signature=Signature(dtype=[dtype('float32')], shape=[(11,)]), action_signature=Signature(dtype=[dtype('float32')], shape=[(3,)]), reward_signature=Signature(dtype=[dtype('float32')], shape=[(1,)]), action_space=<ActionSpace.CONTINUOUS: 1>, action_size=3)
2025-06-21 13:05.51 [debug    ] Fitting observation scaler...  observation_scaler=standard
2025-06-21 13:05.52 [debug    ] Building models...            
2025-06-21 13:05.52 [debug    ] Models have been built.       
2025-06-21 13:05.52 [info     ] Directory is created at d3rlpy_logs/DT_hopper-medium-expert-v2_1_20250621130552
2025-06-21 13:05.52 [info     ] Parameters                     params={'observation_shape': [11], 'action_size': 3, 'config': {'type': 'decision_transformer', 'params': {'batch_size': 64, 'gamma': 0.99, 'observation_scaler': {'type': 'standard', 'params': {'mean': [1.3297109510737133, -0.09838471457011892, -0.54442

Epoch 1/100: 100%|██████████| 1000/1000 [00:27<00:00, 36.31it/s, loss=1.06]

2025-06-21 13:06.19 [info     ] DT_hopper-medium-expert-v2_1_20250621130552: epoch=1 step=1000 epoch=1 metrics={'time_sample_batch': 0.00816592049598694, 'time_algorithm_update': 0.019020316123962402, 'loss': 1.0590533695220947, 'time_step': 0.027313426017761232} step=1000



Epoch 2/100: 100%|██████████| 1000/1000 [00:26<00:00, 37.71it/s, loss=0.524]

2025-06-21 13:06.46 [info     ] DT_hopper-medium-expert-v2_1_20250621130552: epoch=2 step=2000 epoch=2 metrics={'time_sample_batch': 0.008170995473861695, 'time_algorithm_update': 0.017986016511917113, 'loss': 0.5234730297923088, 'time_step': 0.02628284239768982} step=2000


...


Epoch 65/100: 100%|██████████| 1000/1000 [00:27<00:00, 37.03it/s, loss=0.18]


2025-06-21 13:52.11 [info     ] DT_hopper-medium-expert-v2_1_20250621130552: epoch=65 step=65000 epoch=65 metrics={'time_sample_batch': 0.007626987218856811, 'time_algorithm_update': 0.019030782699584962, 'loss': 0.17986808906495572, 'time_step': 0.026779140949249266, 'environment': 1033.8058207117701} step=65000


Epoch 66/100: 100%|██████████| 1000/1000 [00:26<00:00, 37.93it/s, loss=0.18]

2025-06-21 13:52.37 [info     ] DT_hopper-medium-expert-v2_1_20250621130552: epoch=66 step=66000 epoch=66 metrics={'time_sample_batch': 0.007399648427963257, 'time_algorithm_update': 0.018632397413253784, 'loss': 0.1796995084732771, 'time_step': 0.0261525719165802} step=66000



Epoch 67/100:  96%|█████████▌| 960/1000 [00:26<00:01, 36.96it/s, loss=0.179]

In [10]:
import gymnasium as gym
gym.make("Walker2d-v4")

<TimeLimit<OrderEnforcing<PassiveEnvChecker<Walker2dEnv<Walker2d-v4>>>>>

In [21]:
!pip list | grep gymnasium
!pip list | grep mujoco

In [12]:
!ls ../../../../master_thesis/reproducing_decision_transformer/gymnasium/data/

d4rl				  hopper-medium-v2.pkl
download_d4rl_datasets.ipynb	  specs_halfcheetah_v2.json
halfcheetah-medium-expert-v2.pkl  specs_hopper_v2.json
halfcheetah-medium-replay-v2.pkl  specs_walker2d_v2.json
halfcheetah-medium-v2.pkl	  walker2d-medium-replay-v2.pkl
hopper-medium-expert-v2.pkl	  walker2d-medium-v2.pkl
hopper-medium-replay-v2.pkl


In [ ]:
halfcheetah-medium-expert-v2
halfcheetah-medium-replay-v2
halfcheetah-medium-v2